<a href="https://colab.research.google.com/github/Annette-1/SIMULACION-II/blob/main/Ley_de_los_grandes_numeros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ley fuerte de los grandes números para una Cadena de Markov

Para una cadena de Markov finita, irreducible en su clase recurrente y aperiódica,
con distribución estacionaria $\pi$ ($\pi P = \pi$), la ley fuerte de los grandes
números (teorema ergódico) afirma que, con probabilidad $1$:

1) La FRECUENCIA RELATIVA de visitas a cada estado $j$ converge a $\pi_j$:
 $$\frac{1}{n} * \# {k ≤ n : X_k = j}$$  ---> $ \pi_j$        (n -> infinito)

2) El PROMEDIO DE GASTOS (costos) converge al costo esperado bajo pi:
$$ \frac{1}{n} * \sum_{k=1}^n C(X_k)  --->  \sum_j \pi_j * C(j)$$



In [89]:
import numpy as np
import random as r

In [90]:
E=["Y","E","H","P"]  # Espacio de estados

In [91]:
P=np.array([
   [0.0, 0.0, 0.5, 0.5],   # Matriz de transición (cada fila suma 1)
   [0.25, 0.25, 0.25, 0.25],
   [0.25, 0.0, 0.25, 0.5],
   [0.25, 0.0, 0.25, 0.5]
])

In [92]:
P.shape

(4, 4)

In [93]:
len(P)

4

In [94]:
C=np.array([3,7,4,4])   # Costo (gasto) por visitar cada estad

#Estado inicial
Actual=0
Gastos=[] # Lista vacia se guardarán los gastos de la trayectoria

**Simulación de la cadena**


In [95]:
def CM (P,X0,n,E):
  L=[]
  for i in range (n):
     # Se elige la fila de P correspondiente al estado actual
    if X0==E[0]:
      seleccion=r.choices(E,weights=P[0],k=1)
    elif X0==E[1]:
      seleccion=r.choices(E,weights=P[1],k=1)
    elif X0==E[2]:
      seleccion=r.choices(E,weights=P[2],k=1)
    else:
      seleccion=r.choices(E,weights=P[3],k=1)
      # NOTA: r.choices regresa una LISTA (p. ej. ["H"]). Se toma [0] para
        # quedarnos con la etiqueta; si no, X0 sería una lista y las
        # comparaciones X0 == E[j] nunca serían verdaderas.

    X0=seleccion
    L.append(seleccion)
  return L

In [96]:
CM(P, X0="E", n=1, E=["Y","E","H","P"])

[['Y']]

Definimos las funciones para trabajar con probabilidades

In [97]:
def frecuencias(L, E):
    #Frecuencia relativa de cada estado en la trayectoria L (vector de probabilidades)."""
    n = len(L)
    return np.array([L.count(e) / n for e in E])

Calculamos la Distribución estacionaria $\pi$: resuelve $\pi P  = \pi $ con  $\sum(\pi) = 1$.
Se plantea$(P^T - I) \pi = 0$ junto con la fila de unos para la normalización.

In [98]:
def estacionaria(P):
    m = len(P)
    A = np.vstack([P.T - np.eye(m), np.ones(m)])
    b = np.zeros(m + 1)
    b[-1] = 1
    pi, *_ = np.linalg.lstsq(A, b, rcond=None)
    pi = np.clip(pi, 0, None)          # elimina el -0.0000 por error numérico
    return pi / pi.sum()

In [99]:
def imprime_probs(titulo, p, E):
    # esto imprime un vector de probabilidades con su estado."""
    print(titulo)
    for e, pj in zip(E, p):
        print(f"   P({e}) = {pj:.4f}")


In [101]:
if __name__ == "__main__":
    r.seed(42)  # semilla para que los resultados sean reproducibles

    # Matriz de transición como probabilidades
    print("Matriz de transición P (probabilidades):")
    print("        " + "".join(f"{e:>8}" for e in E))
    for i, e in enumerate(E):
        print(f"   {e:>3}  " + "".join(f"{P[i, j]:8.2f}" for j in range(len(E))))
    print()

    # Valor teórico al que deben converger las frecuencias
    pi = estacionaria(P)
    gasto_teorico = float(np.dot(pi, C))
    imprime_probs("Distribución estacionaria teórica (pi):", pi, E)
    print(f"   Gasto promedio teórico  sum(pi_j * C_j) = {gasto_teorico:.4f}\n")

    # Simulación con n creciente: ley de los grandes números
    print("Simulación (X0 = 'E'): frecuencias relativas y gasto promedio")
    print("-" * 66)
    print(f"{'n':>8} | {'P(Y)':>7} {'P(E)':>7} {'P(H)':>7} {'P(P)':>7} | {'Gasto prom.':>11}")
    print("-" * 66)

    for n in [10, 100, 1000, 10000, 100000]:
        L = CM(P, X0="E", n=n, E=E)
        f = frecuencias(L, E)

        # Gastos de la trayectoria: costo del estado visitado en cada paso
        Gastos = [C[E.index(estado[0])] for estado in L]
        gasto_prom = np.mean(Gastos)

        print(f"{n:>8} | {f[0]:7.4f} {f[1]:7.4f} {f[2]:7.4f} {f[3]:7.4f} | {gasto_prom:11.4f}")

    print("-" * 66)
    print(f"{'teórico':>8} | {pi[0]:7.4f} {pi[1]:7.4f} {pi[2]:7.4f} {pi[3]:7.4f} | {gasto_teorico:11.4f}")

Matriz de transición P (probabilidades):
               Y       E       H       P
     Y      0.00    0.00    0.50    0.50
     E      0.25    0.25    0.25    0.25
     H      0.25    0.00    0.25    0.50
     P      0.25    0.00    0.25    0.50

Distribución estacionaria teórica (pi):
   P(Y) = 0.2000
   P(E) = 0.0000
   P(H) = 0.3000
   P(P) = 0.5000
   Gasto promedio teórico  sum(pi_j * C_j) = 3.8000

Simulación (X0 = 'E'): frecuencias relativas y gasto promedio
------------------------------------------------------------------
       n |    P(Y)    P(E)    P(H)    P(P) | Gasto prom.
------------------------------------------------------------------
      10 |  0.0000  0.0000  0.0000  0.0000 |      3.6000
     100 |  0.0000  0.0000  0.0000  0.0000 |      3.7300
    1000 |  0.0000  0.0000  0.0000  0.0000 |      3.7590
   10000 |  0.0000  0.0000  0.0000  0.0000 |      3.7527
  100000 |  0.0000  0.0000  0.0000  0.0000 |      3.7483
------------------------------------------------------

Otra forma

In [ ]:

import random
import numpy as np # Ensure numpy is imported for np.random.choice

# Definimos los estados y sus precios
estados = ['Yogurt', 'Ensalada', 'Hamburguesa', 'Pizza']
precios = {'Yogurt': 3, 'Ensalada': 7, 'Hamburguesa': 4, 'Pizza': 4}

# Matriz de transición (probabilidades de pasar de un estado a otro)
# El orden de las filas/columnas es: [Yogurt, Ensalada, Hamburguesa, Pizza]
matriz_transicion = {
    'Yogurt': [0.0, 0.0, 0.5, 0.5],
    'Ensalada': [0.25, 0.25, 0.25, 0.25],
    'Hamburguesa': [0.25, 0.0, 0.25, 0.5],
    'Pizza': [0.25, 0.0, 0.25, 0.5]
}

def simular_gasto(dias):
    # Elegimos un estado inicial al azar (usando choice)
    estado_actual = random.choice(estados)

    gasto_total = 0

    for _ in range(dias):
        # Sumamos el precio de la comida de hoy
        gasto_total += precios[estado_actual]

        # Determinamos la próxima comida usando np.random.choice con el argumento p
        probabilidades = matriz_transicion[estado_actual]
        estado_actual = np.random.choice(estados, p=probabilidades)

    return gasto_total / dias

# Simulamos 100,000 días para obtener un promedio preciso
dias_simulados = 100000
gasto_promedio = simular_gasto(dias_simulados)

print(f"Simulación con {dias_simulados} días:")
print(f"Gasto promedio a largo plazo: ${gasto_promedio:.2f}")